# Learning Theory & Generalization

Companion notebook for the [Learning Theory lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/06-learning-theory).

**The idea in one sentence.** What we actually care about is *test* error, but we
only ever optimise *training* error — learning theory studies the **generalization
gap** between them, and why it depends on model **capacity** and dataset **size**.

The two forces at work:

- **Bias–variance:** low-capacity models underfit (high bias, they can't represent
  the truth); high-capacity models overfit (high variance, they memorise noise).
  Test error is **U-shaped** in capacity.
- **More data closes the gap:** with enough examples even a flexible model
  generalises, because it can no longer memorise its way out.

We fit polynomials of growing degree from scratch, **validate the U-curve and the
bias–variance decomposition directly**, then cover the gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

## 1 — A noisy true function

Data is a smooth function plus noise. We fit polynomials of increasing degree (increasing capacity)
and measure error on a held-out test set.

In [ ]:
def true_f(x):
    return np.sin(1.5 * x)

def make_data(n, noise=0.25, seed=0):
    r = np.random.default_rng(seed)
    x = r.uniform(-3, 3, n)
    return x, true_f(x) + r.normal(0, noise, n)

x_tr, y_tr = make_data(20, seed=1)
x_te, y_te = make_data(500, seed=2)

def fit_poly(x, y, deg):
    return np.polyfit(x, y, deg)

def mse(coef, x, y):
    return np.mean((np.polyval(coef, x) - y) ** 2)

## 2 — The generalization gap and the U-curve

As degree grows, training error keeps falling (the model fits its sample ever tighter) but test
error bottoms out and then rises — overfitting. The gap between the two curves is the
generalization gap.

In [ ]:
degrees = range(1, 16)
train_err, test_err = [], []
for d in degrees:
    c = fit_poly(x_tr, y_tr, d)
    train_err.append(mse(c, x_tr, y_tr))
    test_err.append(mse(c, x_te, y_te))

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(list(degrees), train_err, 'o-', color='#2dd4bf', label='training error')
ax.plot(list(degrees), test_err, 's-', color='#fb7185', label='test error')
best = list(degrees)[int(np.argmin(test_err))]
ax.axvline(best, ls='--', color='#888', label=f'best degree = {best}')
ax.set_yscale('log'); ax.set_xlabel('polynomial degree (capacity)'); ax.set_ylabel('MSE (log)')
ax.set_title('Training error falls forever; test error is U-shaped (bias-variance)')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'best degree by test error: {best}')

### Validate: test error is U-shaped, and capacity trades bias for variance

Two checks. First, the best degree by test error is **interior** (neither the
simplest nor the most complex model wins) — the signature of the bias–variance
tradeoff. Second, we decompose error across many training sets: **variance grows**
with degree while **bias falls** initially, exactly as theory predicts.

In [ ]:
# 1. the U-curve minimum is interior
assert 1 < best < 15, 'best capacity should be interior (bias-variance U-curve)'
print(f'best degree = {best} (interior -> genuine U-curve, not monotone)')

# 2. bias-variance decomposition over many resampled training sets
xg = np.linspace(-3, 3, 100); truth = true_f(xg)
def bias_var(deg, n_sets=60):
    preds = np.array([np.polyval(fit_poly(*make_data(20, seed=100 + s), deg), xg) for s in range(n_sets)])
    bias2 = np.mean((preds.mean(0) - truth) ** 2)
    var = np.mean(preds.var(0))
    return bias2, var

b1, v1 = bias_var(1); b4, v4 = bias_var(4); b12, v12 = bias_var(12)
print(f'\ndegree  1: bias^2={b1:8.3f}  variance={v1:10.3f}')
print(f'degree  4: bias^2={b4:8.3f}  variance={v4:10.3f}')
print(f'degree 12: bias^2={b12:8.3f}  variance={v12:10.3f}')
assert b1 > b4, 'higher capacity reduces bias'
assert v1 < v4 < v12, 'higher capacity increases variance'
print('\n✅ interior U-curve; capacity trades falling bias for rising variance')

## 3 — More data shrinks the gap

Fix a high-capacity model (degree 12) and grow the training set. The generalization gap
(test − train error) shrinks as n increases — formalizing 'more data generalizes better'.

In [ ]:
sizes = [12, 20, 40, 80, 160, 320]
gaps = []
for n in sizes:
    xs, ys = make_data(n, seed=7)
    c = fit_poly(xs, ys, 12)
    gaps.append(mse(c, x_te, y_te) - mse(c, xs, ys))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sizes, gaps, 'o-', color='#818cf8')
ax.set_xlabel('training set size n'); ax.set_ylabel('generalization gap (test − train MSE)')
ax.set_title('The generalization gap shrinks with more data')
ax.set_xscale('log'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
for n, g in zip(sizes, gaps):
    print(f'n={n:3d}: gap={g:.3f}')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **capacity ≠ parameter count alone** | regularization, architecture, and data shape the *effective* capacity |
| **the classical U-curve isn't the end** | double descent: error can fall again past interpolation (demo) |
| **test set reuse** | tuning on the test set relearns the gap you were trying to measure — hold out a final set |
| **high-degree polyfit is unstable** | numerically ill-conditioned; the variance blow-up is partly numeric |
| **i.i.d. assumption** | generalization bounds assume train/test from the same distribution; drift breaks them |

Demo: double descent — test error peaks at the interpolation threshold, then
descends again.

In [ ]:
# Double descent: the classical U-curve is not the whole story. As capacity grows past
# the interpolation threshold (enough params to fit the data exactly), modern models can
# see test error FALL AGAIN. We sketch it with ridge-regularized random features.
def rff_test_error(n_features, n_train=20, lam=1e-3, seed=0):
    r = np.random.default_rng(seed)
    xtr, ytr = make_data(n_train, seed=seed)
    xte, yte = x_te, y_te
    W = r.normal(0, 1.0, n_features); b = r.uniform(0, 2*np.pi, n_features)
    Ptr = np.cos(np.outer(xtr, W) + b); Pte = np.cos(np.outer(xte, W) + b)
    A = Ptr.T @ Ptr + lam * np.eye(n_features)
    w = np.linalg.solve(A, Ptr.T @ ytr)
    return np.mean((Pte @ w - yte) ** 2)

widths = [2, 5, 10, 18, 20, 25, 40, 80, 200]
errs = [np.mean([rff_test_error(p, seed=s) for s in range(20)]) for p in widths]
for p, e in zip(widths, errs):
    marker = '  <- interpolation threshold (~n_train=20)' if p == 20 else ''
    print(f'width {p:>3}: test MSE = {e:.3f}{marker}')
print('\nError peaks where #features ~ #samples, then DESCENDS again in the over-parameterized regime.')

## ✏️ Your turn

**Exercise.** Implement `generalization_gap(coef, x_tr, y_tr, x_te, y_te)` (test MSE − train MSE)
and `best_capacity(x_tr, y_tr, x_te, y_te, degrees)` returning the polynomial degree with the lowest
*test* error — the model-selection decision the U-curve is all about. Reuse `fit_poly` and `mse`.

In [ ]:
def generalization_gap(coef, x_tr, y_tr, x_te, y_te):
    # TODO(you): test MSE minus train MSE for the fitted coefficients
    return ...

def best_capacity(x_tr, y_tr, x_te, y_te, degrees):
    # TODO(you): fit each degree, return the one with the lowest test MSE
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
c = fit_poly(x_tr, y_tr, 12)
g = generalization_gap(c, x_tr, y_tr, x_te, y_te)
assert np.isclose(g, mse(c, x_te, y_te) - mse(c, x_tr, y_tr))
assert g > 0                                        # a high-capacity fit generalizes worse than it trains
bd = best_capacity(x_tr, y_tr, x_te, y_te, range(1, 16))
assert bd == best                                   # matches the U-curve minimum
# an underfit (degree 1) has higher test error than the best degree
assert mse(fit_poly(x_tr, y_tr, 1), x_te, y_te) > mse(fit_poly(x_tr, y_tr, bd), x_te, y_te)
print(f'\u2713 gap and model selection correct (best degree = {bd})')

<details>
<summary>Solution</summary>

```python
def generalization_gap(coef, x_tr, y_tr, x_te, y_te):
    return mse(coef, x_te, y_te) - mse(coef, x_tr, y_tr)

def best_capacity(x_tr, y_tr, x_te, y_te, degrees):
    return min(degrees, key=lambda d: mse(fit_poly(x_tr, y_tr, d), x_te, y_te))
```

Training error always favors the most complex model, so you must select capacity on held-out data —
the whole reason validation sets exist. The best degree balances bias (too simple) against variance
(too flexible).

</details>

## Key takeaways

- **We optimise training error but care about test error;** the gap between them is
  what learning theory bounds.
- **Test error is U-shaped in capacity:** the best model is interior — we verified
  it and the underlying bias(down)/variance(up) tradeoff directly.
- **More data shrinks the gap** — a flexible model can't memorise its way out once
  there are enough examples.
- **Modern twist — double descent:** past the interpolation threshold, test error
  can fall *again*, which is why huge over-parameterised nets still generalise.